# Char vs Word: RNN vs Transformer Language Models

**Assignment:** Architectural Deep Dive — Transformers vs. RNNs  
**Goal:** Build, train, and benchmark four models (CharRNN, CharTransformer, WordRNN, WordTransformer) on a Wikipedia corpus (~55k characters).

Pipeline:
1. Setup
2. Corpus Acquisition
3. Tokenization Pipelines
4. Model Definitions
5. Training (loss + perplexity)
6. Inference & Temperature Scaling
7. Export Results (`results/pic`, `results/data`)


## 1. Setup

Import libraries, set random seed, and select device (`cuda` if available, else `cpu`).


In [ ]:
# 导入实验所需的标准库与第三方库
import os
import re
import json
import math
import random
from pathlib import Path
from collections import Counter

import requests
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

print("Imports OK")


In [ ]:
# 固定随机种子，保证实验可复现
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 自动选择 GPU（云端 Colab 通常有 cuda）或 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")


In [ ]:
# 创建输出目录：图表 -> results/pic，关键数值 -> results/data
# 云端运行时会在当前工作目录下生成这两个文件夹
PIC_DIR = Path("results/pic")
DATA_DIR = Path("results/data")
PIC_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Output dirs:", PIC_DIR.resolve(), DATA_DIR.resolve())


## 2. Corpus Acquisition

**What:** Fetch plain text from Wikipedia API and clean it.  
**Input:** Article title(s), target length ≈ 55,000 characters.  
**Output:** Cleaned `raw_text` string.


In [ ]:
# Fetch one Wikipedia article as plain text


In [ ]:
def get_wikipedia_text(title="Germany"):
    """通过 Wikipedia API 拉取一篇英文维基页面的纯文本并做基础清洗。"""
    url = (
        "https://en.wikipedia.org/w/api.php"
        f"?action=query&prop=extracts&explaintext=1&titles={title}&format=json"
    )
    # Wikipedia 要求带 User-Agent，否则请求会被拒绝
    headers = {
        "User-Agent": "CharVSWordGPT/1.0 (Educational assignment; cloud notebook)"
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    pages = response.json()["query"]["pages"]
    extract = list(pages.values())[0].get("extract", "")

    # 去掉 == Section == 这类标题标记，并把换行压成空格
    extract = re.sub(r"==+.*?==+", " ", extract)
    extract = re.sub(r"\s+", " ", extract).strip()
    return extract


In [ ]:
# Build a longer corpus by concatenating articles if needed


In [ ]:
def build_corpus(titles=None, min_chars=55000):
    """按需拼接多篇维基文章，直到达到约 55,000 字符。"""
    if titles is None:
        # 默认主题列表，可按需要增删
        titles = [
            "Germany",
            "Berlin",
            "European Union",
            "World War II",
            "German language",
        ]
    parts = []
    for title in titles:
        text = get_wikipedia_text(title)
        parts.append(text)
        total = sum(len(p) for p in parts)
        print(f"  + {title}: {len(text)} chars | cumulative={total}")
        if total >= min_chars:
            break
    return " ".join(parts)


In [ ]:
# 拉取并清洗语料（目标约 55,000 characters）
raw_text = build_corpus(min_chars=55000)
print(f"Corpus length: {len(raw_text)} characters")
print("Preview:", raw_text[:300], "...")


In [ ]:
# 若单次拉取仍不足，再补几篇（可按需修改标题列表）
if len(raw_text) < 55000:
    extra = build_corpus(
        titles=["Munich", "Frankfurt", "Hamburg", "Cologne", "Dresden"],
        min_chars=55000 - len(raw_text),
    )
    raw_text = (raw_text + " " + extra).strip()
    print(f"Augmented corpus length: {len(raw_text)}")

assert len(raw_text) >= 50000, "Corpus too short; add more Wikipedia titles."
print("Corpus ready.")


## 3. Tokenization Pipelines

Two parallel pipelines:

| Pipeline | Token unit | Notes |
|----------|------------|-------|
| Character-level | each char | small vocab, long sequences |
| Word-level | word / punctuation | frequency-filtered vocabulary |

**Output:** `char_encoded`, `word_encoded`, and lookup dicts.


In [ ]:
# --- PIPELINE A: Character-level tokenization ---
# 每个字符映射为一个整数索引
char_vocab = sorted(list(set(raw_text)))
char_to_int = {ch: i for i, ch in enumerate(char_vocab)}
int_to_char = {i: ch for i, ch in enumerate(char_vocab)}
char_encoded = np.array([char_to_int[ch] for ch in raw_text], dtype=np.int64)

print(f"Char vocab size: {len(char_vocab)}")
print(f"Char sequence length: {len(char_encoded)}")
print("Sample chars:", char_vocab[:20])


In [ ]:
# --- PIPELINE B: Word-level tokenization with frequency filter ---
# 用正则拆成“单词”或“标点”，再按词频过滤稀有词

MIN_WORD_FREQ = 2  # 出现次数 < 2 的词并入 <UNK>

word_tokens_raw = re.findall(r"\w+|[^\w\s]", raw_text, re.UNICODE)
freq = Counter(word_tokens_raw)

# 保留高频词；低频词用特殊符号 <UNK> 替代
kept_words = sorted([w for w, c in freq.items() if c >= MIN_WORD_FREQ])
word_vocab = ["<UNK>"] + kept_words
word_to_int = {w: i for i, w in enumerate(word_vocab)}
int_to_word = {i: w for i, w in enumerate(word_vocab)}

def word_to_id(w):
    return word_to_int.get(w, word_to_int["<UNK>"])

word_encoded = np.array([word_to_id(w) for w in word_tokens_raw], dtype=np.int64)

print(f"Raw word tokens: {len(word_tokens_raw)}")
print(f"Unique raw types: {len(freq)}")
print(f"Filtered word vocab size: {len(word_vocab)} (min_freq={MIN_WORD_FREQ})")
print(f"Word sequence length: {len(word_encoded)}")
print(f"UNK rate: {(word_encoded == 0).mean():.2%}")


In [ ]:
def generate_batches(data, batch_size, seq_length):
    """把一维 token 序列切成 (x, y) batch；y 是向右平移 1 位的下一个 token。"""
    total = batch_size * seq_length
    n_batches = len(data) // total
    if n_batches == 0:
        raise ValueError("Not enough data for the chosen batch_size/seq_length.")

    arr = data[: n_batches * total].reshape((batch_size, -1))
    # 注意：最后一段不够 seq_length+1 时停止，避免越界
    for n in range(0, arr.shape[1] - seq_length, seq_length):
        x = arr[:, n : n + seq_length]
        y = arr[:, n + 1 : n + seq_length + 1]
        yield torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


# 快速检查 batch 形状
_xb, _yb = next(generate_batches(char_encoded, batch_size=8, seq_length=32))
print("Batch x:", tuple(_xb.shape), "Batch y:", tuple(_yb.shape))
